# STAT 764 · Lab 8 — Does fixing the numbers pay?

**Assigned Thu Nov 5 · Due Tue Nov 10, 11:59 pm · 25 points**

Individual work. About an hour, and the whole notebook runs in under a minute. You need two tools you
already have: Platt scaling from Meeting 12 (a small logistic regression that corrects a model's
probabilities) and the paired bootstrap from Meeting 18 (is a gap between two models real?).

Same rules as before: `git pull`, copy into `work/` before editing, Restart & Run All before
submitting, upload the `.ipynb` to Canvas.

**How it is graded.** Complete or incomplete, part by part. A part earns its points when everything it
asks for is in your notebook and has run. A written answer earns its points when it points at one of your
numbers. Every split and seed is fixed, so your numbers should match mine. If one does not, say so in a
sentence.

Meeting 18's scoreboard left a question open. TabICL ranked the patients best, but its numbers run low: it
said 9.8% when 11.1% came back. That matters because of Meeting 10's \$150 home visit, which goes to every
patient whose chance is at least 0.169. Numbers that run low send out too few visits.

**So fix the numbers. Does the hospital make more money?**

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import logit
from stat764 import diabetes, load

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# The readmission data from Meetings 9-12, with Meeting 9's split by PATIENT (split 764).
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]  # death, hospice
y_all = (enc["readmitted"] == "<30").astype(int)
NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
X_all = enc[NUM + CAT].astype({c: str for c in CAT})
train, test = next(GroupShuffleSplit(1, test_size=0.25, random_state=764)
                   .split(X_all, y_all, groups=enc["patient_nbr"]))
X_test, y_test = X_all.iloc[test], y_all.iloc[test].to_numpy()
test_patients = enc["patient_nbr"].iloc[test].to_numpy()    # which patient each test stay is

# Meeting 16's sample: a random 15,000 of the training patients, with all their stays.
pool = enc["patient_nbr"].iloc[train].unique()
chosen = np.random.default_rng(16).choice(pool, 15_000, replace=False)
rows = train[np.isin(enc["patient_nbr"].iloc[train], chosen)]

# The model to beat: Meeting 9's logistic regression, exactly as it stood on the scoreboard.
prep = ColumnTransformer([
    ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
])
model = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000))])
logistic = model.fit(X_all.iloc[rows], y_all.iloc[rows].to_numpy()).predict_proba(X_test)[:, 1]

# TabICL's predictions for the test stays, one column per training sample, computed in advance on a GPU.
# sample1 was fitted on the same 15,000 patients as `logistic`.
tabicl = load("tabicl_test_predictions.csv")
raw = tabicl["sample1"].to_numpy()

In [ ]:
READMISSION = 17_700                                   # dollars, Meeting 10
P_STAR = 150 / (0.05 * READMISSION)                    # the $150 visit's break-even: 0.169


def net_per_1000(outcome, probs):
    """Meeting 10's money: dollars saved minus dollars spent per 1,000 stays, visiting probs >= P_STAR."""
    visit = probs >= P_STAR
    return ((visit & (outcome == 1)).sum() * 0.05 * READMISSION - visit.sum() * 150) / len(outcome) * 1000


def calibration_slope(probs, outcome):
    """Meeting 11's slope: 1 = the spread is right; below 1 = too extreme; above 1 = too timid."""
    return LogisticRegression(C=1e6, max_iter=1000).fit(to_logit(probs), outcome).coef_[0, 0]


def to_logit(p):
    """Chances as the one-column table Platt scaling is fitted on."""
    return logit(np.clip(p, 1e-6, 1 - 1e-6)).reshape(-1, 1)


def scores(probs, outcome=y_test):
    """One row of the scoreboard: the order, the numbers, and the money."""
    return {"AUC": roc_auc_score(outcome, probs), "Brier": brier_score_loss(outcome, probs),
            "average said": probs.mean(), "calibration slope": calibration_slope(probs, outcome),
            "share visited": np.mean(probs >= P_STAR), "net per 1,000": net_per_1000(outcome, probs)}


# Meeting 18's paired bootstrap: redraw the test PATIENTS, score both columns on each redraw, keep the gap.
_ids, _which = np.unique(test_patients, return_inverse=True)
_members = np.split(np.argsort(_which), np.cumsum(np.bincount(_which))[:-1])   # each patient's test rows


def paired_gap(p, q, measure=roc_auc_score, draws=300, seed=764):
    """measure(p) - measure(q) on the test stays, and the middle 95% over redraws of the test PATIENTS."""
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(draws):
        b = np.concatenate([_members[i] for i in rng.integers(0, len(_ids), len(_ids))])
        diffs.append(measure(y_test[b], p[b]) - measure(y_test[b], q[b]))
    return measure(y_test, p) - measure(y_test, q), *np.percentile(diffs, [2.5, 97.5])


# Two halves of the test patients. All of a patient's stays land in the same half.
half_A = np.isin(test_patients, np.random.default_rng(8).choice(_ids, len(_ids) // 2, replace=False))
half_B = ~half_A

print(f"test: {len(y_test):,} stays · came back {y_test.mean():.1%} · break-even {P_STAR:.3f}")
print(f"half A: {half_A.sum():,} stays · half B: {half_B.sum():,} stays\n")
print(pd.DataFrame({"logistic regression": scores(logistic), "TabICL": scores(raw)}).T.round(4).to_string())

Those two rows are Meeting 18's scoreboard, to check you are starting where we stopped.

## How we learn the fix

Meeting 12's rule: a fix has to be learned on patients the model was not fitted on, and judged on patients
the fix was not learned on. TabICL never saw the test patients, so the first half of that rule is met. For
the second half we use the two halves:

- learn the fix on half A, and use it on half B;
- learn the fix on half B, and use it on half A.

Every test stay then gets a fixed number from a fix learned on **other** patients, and we can score all
24,706 stays, which keeps the scoreboard comparable. In real use you would set patients aside for this
before touching the test set. We use the test patients only because they are the only ones we have TabICL
predictions for.

---

## Part 1 · Fix the numbers (5 points)

**(i)** Write `cross_fix(p)`. It takes one column of predictions for the test stays and returns the fixed
column, in the same order. A Platt fix is one logistic regression, fitted on the logit of the chances:

```python
platt = LogisticRegression(C=1e6, max_iter=1000).fit(to_logit(p[half_A]), y_test[half_A])
on_B = platt.predict_proba(to_logit(p[half_B]))[:, 1]
```

**(ii)** Set `fixed = cross_fix(raw)`, then print a three-row table with `scores`: logistic regression,
TabICL, and TabICL fixed.

**(iii)** Print the AUC of `raw` and of `fixed` on half A alone (`y_test[half_A]`). You will need it for
question (a).

In [ ]:
# YOUR CODE HERE

## Part 2 · Follow the money (5 points)

**(i)** Is the money gap real? Print two lines with `paired_gap(..., net_per_1000)`: `fixed` against `raw`,
and `fixed` against `logistic`. Each gives the gap and the middle 95% of the redraws.

**(ii)** The fix moves some stays across the line. Set

```python
crossed = (fixed >= P_STAR) & (raw < P_STAR)
```

and print how many stays crossed, how many went the other way, and the share of the crossed stays that
came back. Put `P_STAR` beside that share.

**(iii)** Price those visits. A visit prevents 5% of a \$17,700 readmission and costs \$150, so per visit
the crossed stays earned `rate * 0.05 * READMISSION - 150`, where `rate` is the share you just found.
Multiply by the number that crossed, then divide by `len(y_test)` and multiply by 1,000. Print that beside
the change in `net per 1,000` from your Part 1 table.

In [ ]:
# YOUR CODE HERE

## Part 3 · Five samples, not one (5 points)

One column is one draw. `tabicl` has five, one for each training sample of 15,000 patients. For each column,
use `cross_fix` and `scores`, and print a five-row table with these columns:

| | |
|---|---|
| average said | unfixed, then fixed |
| calibration slope | unfixed, then fixed |
| share visited | unfixed, then fixed |
| net per 1,000 | unfixed, then fixed |

In [ ]:
# YOUR CODE HERE

## Part 4 · Written (10 points)

Two or three sentences each, and point at your numbers.

**(a)** (3 points) A Platt fix only stretches and shifts the chances, so it cannot change the order of the
stays. Yet the AUC moved a little between TabICL and TabICL fixed. Use your half-A check from Part 1 to say
why.

**(b)** (4 points) What happened to the numbers, and what happened to the money? Explain the money with the
stays that crossed the line. What does "break-even" say about a stay sitting near 0.169?

**(c)** (3 points) Go back to the model you recommended in Meeting 18's studio for the \$150 visit. Does
this lab change your recommendation? Then name one use of these predictions where TabICL's unfixed numbers
would do real harm, and say why.

### Your answers

*(double-click to edit)*

**(a)**

**(b)**

**(c)**

---

## Before you submit

- [ ] **Run → Restart Kernel and Run All Cells**
- [ ] Part 1 has `cross_fix`, the three-row table, and the two half-A AUCs
- [ ] Part 2 has both money gaps with their intervals, the counts and share for the crossed stays beside
      `P_STAR`, and the priced visits beside the change in `net per 1,000`
- [ ] Part 3 has the five-row table
- [ ] All three written answers are filled in, each pointing at a number
- [ ] Uploaded the `.ipynb` to Canvas

**25 points.**